# CR-FIQA 기반 조건부 threshold calibration

이 노트북은 완료된 SurvFace Step-4 run(기본값: ArcFace)을 **읽기 전용 입력**으로 사용해 다음 세 가지를 비교합니다.

1. `global_empirical`: 기존 calibration 전체에서 구한 전역 threshold
2. `global_safe`: calibration 내부 fit/safety 분할을 거친 보수적 전역 threshold
3. `fiqa_2bin_conservative_shrunk_safe`: CR-FIQA Low/High 그룹별 부분 풀링 + held-out safety threshold

Saliency의 연구상 위치는 바꾸지 않습니다.

- **1차 목적:** 원본 FR 모델의 공간적 인식 근거가 압축에 따른 embedding distortion, score/rank 변화, threshold crossing과 어떻게 연관되는지 분석
- **2차 목적:** FIQA만으로 설명되지 않는 threshold 불안정성을 saliency가 추가로 설명하는지 검증

현재 SurvFace saliency는 test probe에만 존재하므로, 1차 분석은 가능하지만 FIQA+Saliency threshold 학습은 calibration saliency가 확보될 때까지 누수 방지 게이트가 차단합니다. 기존 공통 orchestration/report 노트북은 수정하지 않습니다.

In [1]:
# 0. 프로젝트 경로와 공통 import
from __future__ import annotations

import gc
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'research').is_dir() and (candidate / '.git').exists():
            return candidate
    raise RuntimeError('C:\\ronbun 프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.evaluation import (
    assess_saliency_faithfulness_reliability,
    load_selected_faithfulness_artifacts,
    resolve_common_faithfulness_maximum_samples,
)
from research.experiments.fiqa_threshold_calibration import (
    assess_saliency_incremental_readiness,
    join_fiqa_score_artifacts,
    load_calibration_comparison_artifact,
    load_condition_score_artifact,
    load_saliency_primary_diagnostics,
    replay_survface_adc_condition_scores,
    upgrade_condition_score_artifact,
    run_global_vs_fiqa_calibration,
    write_calibration_comparison_artifact,
    write_condition_score_artifact,
)
from research.fiqa import (
    CRFIQA_VARIANTS,
    infer_cr_fiqa_scores,
    load_cr_fiqa,
    load_fiqa_score_artifact,
    materialize_aligned_bundle_score_artifact,
)
from research.runtime.hashing import canonical_sha256, sha256_file

PROJECT_ROOT

WindowsPath('C:/ronbun')

## 1. 실험 계약

- **주 실험:** CR-FIQA(S), Low/High 2분위, PQ `m=128`, ADC exhaustive search
- **민감도 분석:** 주 실험이 끝난 뒤 CR-FIQA(L)만 동일 설정으로 반복
- FIQA cutpoint와 모든 threshold는 calibration에서만 결정하며 test 결과를 보지 않습니다.
- ADC의 점수공간은 `negative_squared_l2_adc`이며 원본 cosine threshold를 재사용하지 않습니다.
- Low/High group 표본 부족 시 전역 threshold로 fallback하고, 충분한 경우에도 전역값과 부분 풀링합니다.
- `high_saliency`/`low_saliency` mask는 intervention/faithfulness 분석용입니다. `random`은 음성 대조군이며 threshold feature로 사용하지 않습니다.
- 완료 run과 checkpoint는 덮어쓰지 않습니다. 모든 결과 쓰기는 별도 플래그로 명시합니다.

In [2]:
# 2. 사용자가 조정하는 유일한 설정 셀
SURVFACE_RUN_CANDIDATES = {
    'arcface': PROJECT_ROOT / 'runs' / 'survface_20260902' / (
        '20260902-R001-61915edf_step4_survface_arcface-7972a704552df378345f'
    ),
    'adaface': PROJECT_ROOT / 'runs' / 'survface_20260830' / (
        '20260830-R001-ec6e5d4a_step4_survface_adaface-4df25b75e065b0b9ed43'
    ),
}
SOURCE_MODEL = 'adaface'
assert SOURCE_MODEL in SURVFACE_RUN_CANDIDATES, 'SOURCE_MODEL은 명시된 완료 run 후보여야 합니다.'
SOURCE_RUN_DIR = SURVFACE_RUN_CANDIDATES[SOURCE_MODEL]
ALIGNED_BUNDLE_DIR = PROJECT_ROOT / 'data' / 'interim' / 'step4' / 'survface' / 'aligned_112'
FIQA_CHECKPOINTS = {
    'S': PROJECT_ROOT / 'models' / 'fiqa' / 'CR-FIQA(S).pth',
    'L': PROJECT_ROOT / 'models' / 'fiqa' / 'CR-FIQA(L).pth',
}
# FIQA_VARIANT = 'S'
FIQA_VARIANT = 'L'
FIQA_BATCH_SIZE = 64
FIQA_SHARD_SIZE = 8192
MODEL_SMOKE_VARIANTS = ('S', 'L')
MODEL_SMOKE_SAMPLE_COUNT = FIQA_BATCH_SIZE

COMPRESSION_PROFILE = 'pq_512_m128_b8'
SEARCH_MODE = 'pq_adc_exhaustive'
TARGET_FPIRS = (0.01, 0.05, 0.10, 0.20, 0.30)
QUALITY_BIN_COUNT = 2
SHRINKAGE_STRENGTH = 200.0
MINIMUM_GROUP_NON_MATED = 100
SAFETY_FRACTION = 0.30
PARTITION_SEED = 8972  # S/L 모두 같은 값으로 고정; test 결과로 seed를 선택하지 않음
METRIC_CONTRACT = 'genuine-score-topk-v2'

RESULT_ROOT = PROJECT_ROOT / 'results' / 'calibration'
VERIFY_CHECKPOINT_HASHES = True
OVERWRITE_OUTPUTS = False

# 긴 계산 및 쓰기는 아래 섹션별로 순서대로 실행하십시오.
# ==================== 모델 사전 점검 ====================
RUN_MODEL_SMOKE = False  # CR-FIQA S/L checkpoint GPU smoke test

# ==================== 1. FIQA 점수 생성 ====================
RUN_FIQA_INFERENCE = False  # 전체 aligned 얼굴의 CR-FIQA 점수 추론 실행
WRITE_FIQA_ARTIFACT = False  # 추론한 FIQA 점수를 artifact로 저장

# ==================== 2. PQ-m128 ADC 점수 재생성 ====================
RUN_SCORE_REPLAY = True  # calibration/test의 PQ-m128 ADC 점수 재계산
WRITE_SCORE_ARTIFACT = True  # 재계산한 ADC condition score를 저장

# ==================== 3. Global/FIQA threshold 비교 ====================
RUN_THRESHOLD_CALIBRATION = True  # Global과 FIQA 조건부 threshold 비교 실행
WRITE_CALIBRATION_ARTIFACT = True  # threshold 비교 결과를 artifact로 저장

# ==================== 4. Saliency 진단 ====================
RUN_PRIMARY_SALIENCY_SUMMARY = False
RUN_SALIENCY_READINESS_CHECK = False

assert FIQA_VARIANT in FIQA_CHECKPOINTS
assert QUALITY_BIN_COUNT == 2, '주 분석은 사전 지정된 Low/High 2분위를 사용합니다.'
for stage_name, run_enabled, write_enabled in (
    ('FIQA inference', RUN_FIQA_INFERENCE, WRITE_FIQA_ARTIFACT),
    ('ADC score replay', RUN_SCORE_REPLAY, WRITE_SCORE_ARTIFACT),
    ('threshold calibration', RUN_THRESHOLD_CALIBRATION, WRITE_CALIBRATION_ARTIFACT),
):
    if write_enabled and not run_enabled:
        raise ValueError(f'{stage_name}: WRITE=True이면 대응하는 RUN도 True여야 합니다.')

### 권장 실행 순서

노트북은 B → C → D 순서로 실행됩니다. 처음부터 모두 수행하려면 각 `RUN_*`/`WRITE_*` 플래그를 함께 `True`로 설정하고 위에서부터 한 번 실행할 수 있습니다. `OVERWRITE_OUTPUTS=False`이면 이미 완료된 artifact는 SHA-256 검증 후 재사용하고, 없는 artifact만 다음 순서로 생성하므로 재실행도 안전합니다.

1. `RUN_MODEL_SMOKE=True` — S/L checkpoint strict-load 및 production batch(기본 64장) GPU 추론
2. `RUN_FIQA_INFERENCE=True`, `WRITE_FIQA_ARTIFACT=True` — 전체 aligned bundle의 CR-FIQA(S) 점수 생성
3. `RUN_SCORE_REPLAY=True`, `WRITE_SCORE_ARTIFACT=True` — 저장되지 않았던 calibration PQ-ADC 검색만 재생하고 기존 threshold와 일치 여부 감사
4. `RUN_THRESHOLD_CALIBRATION=True`, `WRITE_CALIBRATION_ARTIFACT=True` — Global/FIQA 비교 산출
5. 필요할 때만 `FIQA_VARIANT='L'`로 바꾸어 민감도 분석 반복

실행이 중단되면 같은 설정으로 위에서부터 다시 실행합니다. 완료된 단계는 검증 후 건너뛰고, 중단된 FIQA shard부터 재개합니다. 기존 완료 artifact를 의도적으로 다시 만들 때만 `OVERWRITE_OUTPUTS=True`를 사용합니다.

In [3]:
# 3. 경량 preflight: 입력 존재, 모델 크기/hash, run 및 aligned contract
def read_json_object(path: Path) -> dict:
    payload = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(payload, dict):
        raise ValueError(f'JSON object가 아닙니다: {path}')
    return payload


if not SOURCE_RUN_DIR.is_dir() or not (SOURCE_RUN_DIR / 'COMPLETED').is_file():
    raise FileNotFoundError(f'완료된 source run이 없습니다: {SOURCE_RUN_DIR}')
source_manifest = read_json_object(SOURCE_RUN_DIR / 'run_manifest.json')
if source_manifest.get('status') != 'completed':
    raise ValueError('source run status가 completed가 아닙니다.')
if source_manifest.get('config', {}).get('dataset_id') != 'survface':
    raise ValueError('이 노트북의 첫 구현은 SurvFace 전용입니다.')
source_model_uid = str(source_manifest.get('config', {}).get('model_uid', ''))
if not source_model_uid:
    raise ValueError('source run에 model_uid가 없습니다.')

aligned_manifest = read_json_object(ALIGNED_BUNDLE_DIR / 'bundle_manifest.json')
contract = aligned_manifest.get('array_contract', {})
if not (
    contract.get('dtype') == 'uint8'
    and contract.get('layout') == 'nhwc'
    and contract.get('color_order') == 'rgb'
    and contract.get('image_size') == [112, 112]
):
    raise ValueError('CR-FIQA 입력은 uint8 NHWC RGB 112x112 aligned bundle이어야 합니다.')

checkpoint_rows = []
for variant, checkpoint in FIQA_CHECKPOINTS.items():
    spec = CRFIQA_VARIANTS[variant]
    if not checkpoint.is_file():
        raise FileNotFoundError(f'CR-FIQA({variant}) checkpoint가 없습니다: {checkpoint}')
    actual_sha256 = sha256_file(checkpoint) if VERIFY_CHECKPOINT_HASHES else None
    checkpoint_rows.append(
        {
            'variant': variant,
            'architecture': spec.architecture,
            'bytes_match': checkpoint.stat().st_size == spec.expected_bytes,
            'sha256_match': actual_sha256 == spec.expected_sha256 if actual_sha256 else 'not_checked',
            'model_uid': spec.model_uid,
            'license': spec.license_id,
        }
    )
checkpoint_preflight = pd.DataFrame(checkpoint_rows)
if not checkpoint_preflight['bytes_match'].all():
    raise ValueError('checkpoint byte size가 등록된 공식 파일과 다릅니다.')
if VERIFY_CHECKPOINT_HASHES and not checkpoint_preflight['sha256_match'].all():
    raise ValueError('checkpoint SHA-256이 등록된 공식 파일과 다릅니다.')

display(checkpoint_preflight)
display(
    pd.DataFrame(
        [{
            'source_run_id': source_manifest['run_id'],
            'dataset_id': source_manifest['config']['dataset_id'],
            'model_uid': source_manifest['config']['model_uid'],
            'aligned_rows': aligned_manifest.get('counts', {}).get('aligned'),
            'selected_fiqa_variant': FIQA_VARIANT,
        }]
    )
)

,variant,architecture,bytes_match,sha256_match,model_uid,license
0,S,iresnet50,True,True,cr-fiqa-s-b9f457a6f00e0363a0cf,CC-BY-NC-4.0
1,L,iresnet100,True,True,cr-fiqa-l-5fca24736e4f8df5fbfc,CC-BY-NC-4.0


,source_run_id,dataset_id,model_uid,aligned_rows,selected_fiqa_variant
0,20260830-R001-ec6e5d4a,survface,adaface-4df25b75e065b0b9ed43,463341,L


## 4. 선택 단계 A — CR-FIQA S/L GPU smoke test

공식 구조와 state dict를 `strict=True`로 불러오며 CUDA가 없으면 CPU로 조용히 전환하지 않고 중단합니다. 점수는 sigmoid/min-max 변환 없이 checkpoint의 raw scalar 그대로 사용합니다.

In [4]:
smoke_summary = pd.DataFrame()
if RUN_MODEL_SMOKE:
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError('CUDA smoke test가 요청됐지만 torch.cuda.is_available()이 False입니다.')
    aligned_index = pd.read_csv(
        ALIGNED_BUNDLE_DIR / 'aligned_index.csv',
        nrows=MODEL_SMOKE_SAMPLE_COUNT,
        usecols=['sample_id', 'aligned_face_index'],
    )
    aligned_faces = np.load(
        ALIGNED_BUNDLE_DIR / 'aligned_faces.npy', mmap_mode='r', allow_pickle=False
    )
    face_indices = aligned_index['aligned_face_index'].to_numpy(dtype=np.int64)
    smoke_faces = np.asarray(aligned_faces[face_indices])
    smoke_rows = []
    for variant in MODEL_SMOKE_VARIANTS:
        torch.cuda.reset_peak_memory_stats()
        model, spec = load_cr_fiqa(
            FIQA_CHECKPOINTS[variant], variant=variant, device='cuda',
            verify_official_hash=True,
        )
        scores = infer_cr_fiqa_scores(
            model, smoke_faces, batch_size=MODEL_SMOKE_SAMPLE_COUNT, device='cuda'
        )
        smoke_rows.append(
            {
                'variant': variant,
                'model_uid': spec.model_uid,
                'sample_count': len(scores),
                'finite': bool(np.isfinite(scores).all()),
                'score_min': float(scores.min()),
                'score_max': float(scores.max()),
                'cuda_peak_bytes': int(torch.cuda.max_memory_allocated()),
                'device': torch.cuda.get_device_name(0),
            }
        )
        del model, scores
        gc.collect()
        torch.cuda.empty_cache()
    smoke_summary = pd.DataFrame(smoke_rows)
    display(smoke_summary)
else:
    display(Markdown('`RUN_MODEL_SMOKE=False`: checkpoint GPU smoke를 건너뜁니다.'))

`RUN_MODEL_SMOKE=False`: checkpoint GPU smoke를 건너뜁니다.

## 5. 선택 단계 B — 전체 SurvFace CR-FIQA score artifact

모든 aligned face에 scalar quality를 한 번만 계산해 `sample_id, fiqa_score, fiqa_model_uid` 형태로 저장합니다. 전체 입력 bundle과 checkpoint lineage를 manifest에 고정합니다. 각 8,192장 shard를 원자적으로 기록하므로 중단 후 동일 설정으로 다시 실행하면 완료 shard부터 재개합니다. 전체 추론을 요청할 때는 쓰기 플래그도 반드시 켜야 합니다.

In [5]:
selected_spec = CRFIQA_VARIANTS[FIQA_VARIANT]
FIQA_OUTPUT_DIR = RESULT_ROOT / 'fiqa_scores' / 'survface' / selected_spec.model_uid
fiqa_artifact = None
fiqa_stage_action = 'not_run'

if RUN_FIQA_INFERENCE:
    if FIQA_OUTPUT_DIR.exists() and not OVERWRITE_OUTPUTS:
        fiqa_artifact = load_fiqa_score_artifact(FIQA_OUTPUT_DIR)
        fiqa_stage_action = 'reused_verified'
        display(Markdown(f'완료된 FIQA artifact를 검증 후 재사용합니다: `{FIQA_OUTPUT_DIR}`'))
    else:
        if not WRITE_FIQA_ARTIFACT:
            raise ValueError('새 FIQA inference에는 WRITE_FIQA_ARTIFACT=True가 필요합니다.')
        model, loaded_spec = load_cr_fiqa(
            FIQA_CHECKPOINTS[FIQA_VARIANT], variant=FIQA_VARIANT, device='cuda',
            verify_official_hash=True,
        )
        fiqa_artifact = materialize_aligned_bundle_score_artifact(
            ALIGNED_BUNDLE_DIR,
            FIQA_OUTPUT_DIR,
            model=model,
            model_uid=loaded_spec.model_uid,
            checkpoint_sha256=model.cr_fiqa_checkpoint_sha256,
            variant=loaded_spec.variant,
            batch_size=FIQA_BATCH_SIZE,
            shard_size=FIQA_SHARD_SIZE,
            device='cuda',
            overwrite=OVERWRITE_OUTPUTS,
        )
        fiqa_stage_action = 'computed_written'
        del model
        gc.collect()
else:
    if FIQA_OUTPUT_DIR.is_dir():
        fiqa_artifact = load_fiqa_score_artifact(FIQA_OUTPUT_DIR)
        fiqa_stage_action = 'loaded_verified'

if fiqa_artifact is not None:
    expected_fiqa = {
        'dataset_id': 'survface',
        'fiqa_model_uid': selected_spec.model_uid,
        'checkpoint_sha256': selected_spec.expected_sha256,
    }
    mismatches = {
        key: (fiqa_artifact.manifest.get(key), expected)
        for key, expected in expected_fiqa.items()
        if fiqa_artifact.manifest.get(key) != expected
    }
    if mismatches:
        raise ValueError(f'FIQA artifact가 현재 설정과 다릅니다: {mismatches}')

if fiqa_artifact is None:
    display(Markdown(f'FIQA artifact 대기 중: `{FIQA_OUTPUT_DIR}`'))
else:
    display(pd.DataFrame([fiqa_artifact.manifest['score_summary']]))
    display(fiqa_artifact.scores.head())

,minimum,median,maximum,mean
0,-0.09688,0.562348,2.174285,0.626438


,sample_id,aligned_face_index,aligned_content_sha256,fiqa_score,fiqa_model_uid
0,survface:train:100:100_cam1_1,0,9e9420b9c0f32c3385712030093558439e48e4c419c448...,0.497116,cr-fiqa-l-5fca24736e4f8df5fbfc
1,survface:train:100:100_cam2_1,1,03f47e7734336617b9f673ae84056185b0d152d327983e...,0.516596,cr-fiqa-l-5fca24736e4f8df5fbfc
2,survface:train:100:100_cam3_1,2,c56689c351b775928adf7f1cc51137e7ac8b5e353dab2a...,0.399595,cr-fiqa-l-5fca24736e4f8df5fbfc
3,survface:train:100:100_cam4_1,3,9f44c44a593492e85960d7074c1151bdeb743ce037169f...,1.243854,cr-fiqa-l-5fca24736e4f8df5fbfc
4,survface:train:100:100_cam5_1,4,35ebf8e5bb132ff0339c886824ceab00180e765464708f...,1.075207,cr-fiqa-l-5fca24736e4f8df5fbfc


## 6. 선택 단계 C — calibration PQ-ADC score만 재생

완료된 Step-4 run은 test retrieval core를 보존했지만 calibration query의 개별 compressed score는 threshold 산출 뒤 저장하지 않았습니다. 따라서 이 단계는 기존 embedding과 frozen PQ codec을 재사용해 **calibration search만** 재생합니다. 재생한 점수로 기존 다섯 target FPIR threshold가 `1e-12` 이내에서 재현되지 않으면 artifact를 만들지 않습니다.

In [6]:
LEGACY_CONDITION_DIR = (
    RESULT_ROOT / 'condition_scores' / str(source_manifest['run_id'])
    / f'{COMPRESSION_PROFILE}__{SEARCH_MODE}'
)
CONDITION_OUTPUT_DIR = LEGACY_CONDITION_DIR / METRIC_CONTRACT
condition_tables = None
condition_stage_action = 'not_run'

if RUN_SCORE_REPLAY:
    if CONDITION_OUTPUT_DIR.exists() and not OVERWRITE_OUTPUTS:
        condition_tables = load_condition_score_artifact(CONDITION_OUTPUT_DIR)
        condition_stage_action = 'reused_verified'
        display(Markdown(f'완료된 condition score artifact를 검증 후 재사용합니다: `{CONDITION_OUTPUT_DIR}`'))
    else:
        if not WRITE_SCORE_ARTIFACT:
            raise ValueError('새 calibration score replay에는 WRITE_SCORE_ARTIFACT=True가 필요합니다.')
        if (LEGACY_CONDITION_DIR / 'manifest.json').is_file():
            in_memory_tables = upgrade_condition_score_artifact(LEGACY_CONDITION_DIR, SOURCE_RUN_DIR)
            display(Markdown('검증된 v1 calibration 점수와 test ledger로 v2를 생성합니다. 검색/추론 재실행 없음.'))
        else:
            in_memory_tables = replay_survface_adc_condition_scores(
                SOURCE_RUN_DIR, compression_profile=COMPRESSION_PROFILE, search_mode=SEARCH_MODE
            )
        condition_tables = write_condition_score_artifact(
            CONDITION_OUTPUT_DIR, in_memory_tables, overwrite=OVERWRITE_OUTPUTS
        )
        condition_stage_action = 'computed_written'
else:
    if CONDITION_OUTPUT_DIR.is_dir():
        condition_tables = load_condition_score_artifact(CONDITION_OUTPUT_DIR)
        condition_stage_action = 'loaded_verified'

if condition_tables is not None:
    expected_condition = {
        'source_run_id': str(source_manifest['run_id']),
        'model_uid': source_model_uid,
        'compression_profile': COMPRESSION_PROFILE,
        'search_mode': SEARCH_MODE,
    }
    mismatches = {
        key: (condition_tables.manifest.get(key), expected)
        for key, expected in expected_condition.items()
        if condition_tables.manifest.get(key) != expected
    }
    if mismatches:
        raise ValueError(f'condition artifact가 현재 설정과 다릅니다: {mismatches}')

if condition_tables is None:
    display(Markdown(f'condition score artifact 대기 중: `{CONDITION_OUTPUT_DIR}`'))
else:
    display(pd.DataFrame(condition_tables.manifest['global_threshold_reproduction']))
    display(
        pd.DataFrame(
            [{
                'condition_uid': condition_tables.condition_uid,
                'calibration_rows': len(condition_tables.calibration),
                'test_rows': len(condition_tables.test),
                'score_space': condition_tables.manifest['score_space'],
            }]
        )
    )

,target_fpir,persisted_threshold,reproduced_threshold,absolute_difference,exact_match
0,0.01,-0.393827,-0.393827,0.0,True
1,0.05,-0.467569,-0.467569,0.0,True
2,0.10,-0.515315,-0.515315,0.0,True
3,0.20,-0.581630,-0.581630,0.0,True
4,0.30,-0.636949,-0.636949,0.0,True


,condition_uid,calibration_rows,test_rows,score_space
0,compressed-scores-f830df873474e5adcdbb002a,160408,182159,negative_squared_l2_adc


In [7]:
# 7. FIQA를 calibration/test query에 완전한 one-to-one으로 결합
calibration_with_fiqa = None
test_with_fiqa = None
if fiqa_artifact is not None and condition_tables is not None:
    calibration_with_fiqa, test_with_fiqa = join_fiqa_score_artifacts(
        condition_tables, fiqa_artifact
    )
    display(
        pd.DataFrame(
            [
                {
                    'split': 'calibration', 'rows': len(calibration_with_fiqa),
                    'fiqa_min': calibration_with_fiqa['fiqa_score'].min(),
                    'fiqa_median': calibration_with_fiqa['fiqa_score'].median(),
                    'fiqa_max': calibration_with_fiqa['fiqa_score'].max(),
                },
                {
                    'split': 'test', 'rows': len(test_with_fiqa),
                    'fiqa_min': test_with_fiqa['fiqa_score'].min(),
                    'fiqa_median': test_with_fiqa['fiqa_score'].median(),
                    'fiqa_max': test_with_fiqa['fiqa_score'].max(),
                },
            ]
        )
    )
else:
    display(Markdown('FIQA와 condition score artifact가 모두 준비되면 결합합니다.'))

,split,rows,fiqa_min,fiqa_median,fiqa_max
0,calibration,160408,-0.096880,0.560314,2.112883
1,test,182159,-0.044018,0.565888,2.174285


## 8. 선택 단계 D — Global 대 FIQA-conditioned calibration

Calibration을 `identity_id` SHA-256으로 fit/safety에 결정론적으로 분할합니다. FIQA cutpoint는 fit subset에서만 정하며, Low/High threshold는 group별 non-mated score에서 산출한 뒤 표본수 기반 shrinkage와 safety 상한을 적용합니다. Test에서는 고정된 cutpoint/threshold를 한 번만 적용합니다.

In [8]:
CALIBRATION_OUTPUT_DIR = (
    RESULT_ROOT / 'global_vs_fiqa' / str(source_manifest['run_id'])
    / selected_spec.model_uid / f'{COMPRESSION_PROFILE}__{SEARCH_MODE}'
    / METRIC_CONTRACT / f'partition-{PARTITION_SEED}'
)
comparison = None
calibration_stage_action = 'not_run'

if RUN_THRESHOLD_CALIBRATION:
    if CALIBRATION_OUTPUT_DIR.exists() and not OVERWRITE_OUTPUTS:
        comparison = load_calibration_comparison_artifact(CALIBRATION_OUTPUT_DIR)
        calibration_stage_action = 'reused_verified'
        display(Markdown(f'완료된 calibration artifact를 검증 후 재사용합니다: `{CALIBRATION_OUTPUT_DIR}`'))
    else:
        if calibration_with_fiqa is None or test_with_fiqa is None:
            raise RuntimeError('먼저 FIQA 및 condition score artifact를 준비해야 합니다.')
        in_memory_comparison = run_global_vs_fiqa_calibration(
            calibration_with_fiqa,
            test_with_fiqa,
            target_fpirs=TARGET_FPIRS,
            bin_count=QUALITY_BIN_COUNT,
            shrinkage_strength=SHRINKAGE_STRENGTH,
            minimum_group_non_mated=MINIMUM_GROUP_NON_MATED,
            safety_fraction=SAFETY_FRACTION,
            partition_seed=PARTITION_SEED,
            condition_manifest=condition_tables.manifest,
            fiqa_manifest=fiqa_artifact.manifest,
        )
        comparison = (
            write_calibration_comparison_artifact(
                CALIBRATION_OUTPUT_DIR, in_memory_comparison, overwrite=OVERWRITE_OUTPUTS
            )
            if WRITE_CALIBRATION_ARTIFACT
            else in_memory_comparison
        )
        calibration_stage_action = (
            'computed_written' if WRITE_CALIBRATION_ARTIFACT else 'computed_in_memory'
        )
else:
    if CALIBRATION_OUTPUT_DIR.is_dir():
        comparison = load_calibration_comparison_artifact(CALIBRATION_OUTPUT_DIR)
        calibration_stage_action = 'loaded_verified'

if comparison is not None:
    expected_comparison = {
        'condition_uid': condition_tables.condition_uid if condition_tables is not None else None,
        'fiqa_uid': fiqa_artifact.manifest['fiqa_uid'] if fiqa_artifact is not None else None,
        'score_space': condition_tables.manifest['score_space'] if condition_tables is not None else None,
        'metric_contract': METRIC_CONTRACT,
        'condition_manifest_sha256': canonical_sha256(condition_tables.manifest) if condition_tables is not None else None,
        'fiqa_manifest_sha256': canonical_sha256(fiqa_artifact.manifest) if fiqa_artifact is not None else None,
        'target_fpirs': list(TARGET_FPIRS),
        'quality_condition': {
            'column': 'fiqa_score', 'bin_count': QUALITY_BIN_COUNT,
            'cutpoint_source': 'calibration_fit_only',
            'shrinkage_strength': SHRINKAGE_STRENGTH,
            'minimum_group_non_mated': MINIMUM_GROUP_NON_MATED,
        },
        'safety_calibration': {
            'fraction': SAFETY_FRACTION, 'partition_seed': PARTITION_SEED,
            'partition_key': 'sha256(identity_id)', 'partition_unit': 'identity_cluster',
        },
    }
    mismatches = {
        key: (comparison.manifest.get(key), expected)
        for key, expected in expected_comparison.items()
        if expected is None or comparison.manifest.get(key) != expected
    }
    if mismatches:
        raise ValueError(f'calibration artifact가 현재 입력 lineage와 다릅니다: {mismatches}')

if comparison is None:
    display(Markdown(f'calibration comparison 대기 중: `{CALIBRATION_OUTPUT_DIR}`'))
else:
    result_columns = [
        'target_fpir', 'method', 'realized_fpir',
        'fpir_wilson95_low', 'fpir_wilson95_high', 'tpir_at_rank_k',
        'tpir_at_rank_k_wilson95_low', 'tpir_at_rank_k_wilson95_high',
        'target_met_on_test', 'target_met_by_wilson_upper',
    ]
    display(
        comparison.method_summary[result_columns]
        .sort_values(['target_fpir', 'method']).reset_index(drop=True)
    )
    display(comparison.paired_comparisons)

,target_fpir,method,realized_fpir,fpir_wilson95_low,fpir_wilson95_high,tpir_at_rank_k,tpir_at_rank_k_wilson95_low,tpir_at_rank_k_wilson95_high,target_met_on_test,target_met_by_wilson_upper
0,0.01,fiqa_2bin_conservative_shrunk_safe,0.009258,0.008735,0.009811,0.001854,0.001541,0.002230,True,True
1,0.01,global_empirical,0.010005,0.009461,0.010580,0.002069,0.001737,0.002464,False,False
2,0.01,global_safe,0.009989,0.009445,0.010563,0.002069,0.001737,0.002464,True,False
3,0.05,fiqa_2bin_conservative_shrunk_safe,0.045878,0.044717,0.047068,0.006802,0.006177,0.007489,True,True
4,0.05,global_empirical,0.047168,0.045991,0.048373,0.006819,0.006193,0.007507,True,True
5,0.05,global_safe,0.047003,0.045829,0.048207,0.006785,0.006162,0.007472,True,True
6,0.10,fiqa_2bin_conservative_shrunk_safe,0.090072,0.088477,0.091693,0.012942,0.012071,0.013875,True,True
7,0.10,global_empirical,0.092093,0.090481,0.093730,0.012611,0.011752,0.013532,True,True
8,0.10,global_safe,0.091485,0.089878,0.093117,0.012512,0.011656,0.013430,True,True
9,0.20,fiqa_2bin_conservative_shrunk_safe,0.179840,0.177693,0.182007,0.024643,0.023437,0.025910,True,True


,reference_method,candidate_method,target_fpir,score_space,rank_k,metric,reference_successes,candidate_successes,both_successes,total,candidate_minus_reference,paired_bootstrap95_low,paired_bootstrap95_high,confidence_interval_method,resamples,random_seed,metric_contract,resampling_unit,threshold_uncertainty_included,multiple_comparison_adjustment
0,global_empirical,fiqa_2bin_conservative_shrunk_safe,0.01,negative_squared_l2_adc,20,fpir,1218,1127,1026,121736,-0.000748,-0.001027,-0.000468,paired_nonparametric_bootstrap_percentile,2000,8972,genuine-score-topk-v2,query,False,none
1,global_empirical,fiqa_2bin_conservative_shrunk_safe,0.01,negative_squared_l2_adc,20,tpir_at_rank_k,125,112,105,60423,-0.000215,-0.000397,-0.000050,paired_nonparametric_bootstrap_percentile,2000,8972,genuine-score-topk-v2,query,False,none
2,global_safe,fiqa_2bin_conservative_shrunk_safe,0.01,negative_squared_l2_adc,20,fpir,1216,1127,1025,121736,-0.000731,-0.001010,-0.000460,paired_nonparametric_bootstrap_percentile,2000,8972,genuine-score-topk-v2,query,False,none
3,global_safe,fiqa_2bin_conservative_shrunk_safe,0.01,negative_squared_l2_adc,20,tpir_at_rank_k,125,112,105,60423,-0.000215,-0.000397,-0.000050,paired_nonparametric_bootstrap_percentile,2000,8972,genuine-score-topk-v2,query,False,none
4,global_empirical,fiqa_2bin_conservative_shrunk_safe,0.05,negative_squared_l2_adc,20,fpir,5742,5585,4889,121736,-0.001290,-0.001939,-0.000649,paired_nonparametric_bootstrap_percentile,2000,8972,genuine-score-topk-v2,query,False,none
5,global_empirical,fiqa_2bin_conservative_shrunk_safe,0.05,negative_squared_l2_adc,20,tpir_at_rank_k,412,411,363,60423,-0.000017,-0.000331,0.000298,paired_nonparametric_bootstrap_percentile,2000,8972,genuine-score-topk-v2,query,False,none
6,global_safe,fiqa_2bin_conservative_shrunk_safe,0.05,negative_squared_l2_adc,20,fpir,5722,5585,4880,121736,-0.001125,-0.001783,-0.000517,paired_nonparametric_bootstrap_percentile,2000,8972,genuine-score-topk-v2,query,False,none
7,global_safe,fiqa_2bin_conservative_shrunk_safe,0.05,negative_squared_l2_adc,20,tpir_at_rank_k,410,411,362,60423,0.000017,-0.000298,0.000331,paired_nonparametric_bootstrap_percentile,2000,8972,genuine-score-topk-v2,query,False,none
8,global_empirical,fiqa_2bin_conservative_shrunk_safe,0.10,negative_squared_l2_adc,20,fpir,11211,10965,9546,121736,-0.002021,-0.002933,-0.001142,paired_nonparametric_bootstrap_percentile,2000,8972,genuine-score-topk-v2,query,False,none
9,global_empirical,fiqa_2bin_conservative_shrunk_safe,0.10,negative_squared_l2_adc,20,tpir_at_rank_k,762,782,680,60423,0.000331,-0.000132,0.000778,paired_nonparametric_bootstrap_percentile,2000,8972,genuine-score-topk-v2,query,False,none


## 8.1 동일 split의 S/L 직접 비교 · 인물 단위 CI · 품질별 tail 전이
기존 S/L FIQA score와 C의 v2 condition을 사용합니다. 추가 inference나 embedding 추출은 없습니다. TPIR CI는 mated identity cluster를 재표집하며 query 가중 비율을 유지합니다. FPIR는 unknown identity를 알 수 없어 query 단위입니다. 두 CI 모두 threshold/calibration 재적합·gallery 변동·다중 비교를 포함하지 않는 탐색적 결과입니다. fit/safety/test의 그룹 비중과 frozen threshold 초과율을 함께 읽으세요.

In [9]:
from research.experiments.fiqa_priority_diagnostics import (
    run_fiqa_priority_diagnostics, write_fiqa_priority_diagnostics,
)
RUN_PRIORITY_DIAGNOSTICS = True
WRITE_PRIORITY_DIAGNOSTICS = True
CLUSTER_BOOTSTRAP_RESAMPLES = 2000
CLUSTER_BOOTSTRAP_SEED = 8972
priority_diagnostics = None
if WRITE_PRIORITY_DIAGNOSTICS and not RUN_PRIORITY_DIAGNOSTICS:
    raise ValueError('WRITE requires RUN_PRIORITY_DIAGNOSTICS=True')
if RUN_PRIORITY_DIAGNOSTICS:
    if condition_tables is None:
        raise RuntimeError('먼저 C에서 v2 condition artifact를 준비하세요.')
    paired_fiqa = {
        variant: load_fiqa_score_artifact(
            RESULT_ROOT / 'fiqa_scores' / 'survface' / CRFIQA_VARIANTS[variant].model_uid
        ) for variant in ('S', 'L')
    }
    priority_diagnostics = run_fiqa_priority_diagnostics(
        condition_tables, paired_fiqa['S'], paired_fiqa['L'],
        target_fpirs=TARGET_FPIRS, partition_seed=PARTITION_SEED,
        safety_fraction=SAFETY_FRACTION, bin_count=QUALITY_BIN_COUNT,
        shrinkage_strength=SHRINKAGE_STRENGTH,
        minimum_group_non_mated=MINIMUM_GROUP_NON_MATED,
        resamples=CLUSTER_BOOTSTRAP_RESAMPLES, bootstrap_seed=CLUSTER_BOOTSTRAP_SEED,
    )
    if WRITE_PRIORITY_DIAGNOSTICS:
        priority_path = write_fiqa_priority_diagnostics(
            RESULT_ROOT / 'fiqa_priority_diagnostics' / str(source_manifest['run_id']),
            priority_diagnostics,
        )
        display(Markdown(f'새 진단 저장: `{priority_path}` (기존 결과 덮어쓰기 금지)'))
    display(priority_diagnostics['method_summary'][[
        'target_fpir', 'method', 'realized_fpir', 'tpir_at_rank_k',
        'tpir_cluster95_low', 'tpir_cluster95_high', 'target_met_on_test',
    ]])
    display(priority_diagnostics['paired_comparisons'])
    display(priority_diagnostics['group_tail_transfer'])
else:
    display(Markdown('S/L 공동 진단 대기: 위 RUN/WRITE 플래그로 명시적으로 실행합니다.'))

새 진단 저장: `C:\ronbun\results\calibration\fiqa_priority_diagnostics\20260830-R001-ec6e5d4a\fiqa-priority-3750761b0430ace01b9b0722` (기존 결과 덮어쓰기 금지)

,target_fpir,method,realized_fpir,tpir_at_rank_k,tpir_cluster95_low,tpir_cluster95_high,target_met_on_test
0,0.01,global_empirical,0.010005,0.002069,0.000769,0.004071,False
1,0.01,global_safe,0.009989,0.002069,0.000769,0.004071,True
2,0.01,fiqa_s,0.009455,0.002003,0.000700,0.004108,True
3,0.01,fiqa_l,0.009258,0.001854,0.000704,0.003601,True
4,0.05,global_empirical,0.047168,0.006819,0.003626,0.011492,True
5,0.05,global_safe,0.047003,0.006785,0.003592,0.011452,True
6,0.05,fiqa_s,0.047102,0.006802,0.003550,0.011596,True
7,0.05,fiqa_l,0.045878,0.006802,0.003689,0.011305,True
8,0.10,global_empirical,0.092093,0.012611,0.007936,0.018957,True
9,0.10,global_safe,0.091485,0.012512,0.007892,0.018842,True


,reference_method,candidate_method,target_fpir,metric,reference_successes,candidate_successes,both_successes,total,candidate_minus_reference,paired_bootstrap95_low,paired_bootstrap95_high,resampling_unit,resamples,random_seed,threshold_uncertainty_included,multiple_comparison_adjustment
0,global_empirical,fiqa_s,0.01,fpir,1218,1151,984,121736,-0.000550,-0.000879,-0.000238,query,2000,8972,False,none
1,global_empirical,fiqa_s,0.01,tpir_at_rank_k,125,121,105,60423,-0.000066,-0.000271,0.000137,mated_identity_cluster,2000,8972,False,none
2,global_safe,fiqa_s,0.01,fpir,1216,1151,983,121736,-0.000534,-0.000871,-0.000230,query,2000,8972,False,none
3,global_safe,fiqa_s,0.01,tpir_at_rank_k,125,121,105,60423,-0.000066,-0.000271,0.000137,mated_identity_cluster,2000,8972,False,none
4,global_empirical,fiqa_l,0.01,fpir,1218,1127,1026,121736,-0.000748,-0.001027,-0.000468,query,2000,8972,False,none
5,global_empirical,fiqa_l,0.01,tpir_at_rank_k,125,112,105,60423,-0.000215,-0.000502,0.000017,mated_identity_cluster,2000,8972,False,none
6,global_safe,fiqa_l,0.01,fpir,1216,1127,1025,121736,-0.000731,-0.001010,-0.000460,query,2000,8972,False,none
7,global_safe,fiqa_l,0.01,tpir_at_rank_k,125,112,105,60423,-0.000215,-0.000502,0.000017,mated_identity_cluster,2000,8972,False,none
8,fiqa_s,fiqa_l,0.01,fpir,1151,1127,961,121736,-0.000197,-0.000501,0.000107,query,2000,8972,False,none
9,fiqa_s,fiqa_l,0.01,tpir_at_rank_k,121,112,100,60423,-0.000149,-0.000550,0.000171,mated_identity_cluster,2000,8972,False,none


,split,quality_group,non_mated_count,non_mated_fraction,false_accept_count,realized_fpir,threshold,score_q99,quality_cutpoints,threshold_fit_on_test,method,target_fpir
0,fit,all,73447,1.000000,733,0.009980,-0.393701,-0.393830,[],False,global_safe,0.01
1,safety,all,25378,1.000000,253,0.009969,-0.393701,-0.394403,[],False,global_safe,0.01
2,test,all,121736,1.000000,1216,0.009989,-0.393701,-0.393795,[],False,global_safe,0.01
3,fit,low,36600,0.498319,359,0.009809,-0.381706,-0.382205,[0.54886115],False,fiqa_s,0.01
4,fit,high,36847,0.501681,330,0.008956,-0.407259,-0.411681,[0.54886115],False,fiqa_s,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
70,fit,high,36567,0.497869,10511,0.287445,-0.689625,-0.406854,[0.5589450499999999],False,fiqa_l,0.30
71,safety,low,12160,0.479155,3589,0.295148,-0.599974,-0.386320,[0.5589450499999999],False,fiqa_l,0.30
72,safety,high,13218,0.520845,3965,0.299970,-0.689625,-0.401907,[0.5589450499999999],False,fiqa_l,0.30
73,test,low,58258,0.478560,16610,0.285111,-0.599974,-0.384822,[0.5589450499999999],False,fiqa_l,0.30


## 8.2 선택 단계 — 동일 cohort의 fit/safety 분할 안정성

새 실행의 20개 seed(0–18, 8972; 기존 완료 결과의 목록은 보존) 전체를 같은 Global-safe/S/L 비교에 사용합니다. Global empirical은 전체 calibration을 쓰므로 seed 불변 대조군입니다. 각 seed의 quality cutpoint·threshold를 calibration에서만 다시 적합하며, test/gallery/codec은 고정합니다. 추가 inference·검색은 없습니다.

seed별 TPIR CI는 공통 인물 단위 bootstrap, FPIR CI는 query 단위 Wilson입니다. seed 사이 최소·사분위·중앙값·최대와 목표 충족 횟수는 **기술통계이며 95% CI나 독립 실험 성공 확률이 아닙니다**. 같은 test를 반복 사용하므로 유리한 seed 선택과 독립 반복실험 주장을 금지합니다. calibration cohort/gallery 교체 및 split+test 결합 CI는 포함하지 않습니다.

기본 RUN/WRITE는 False입니다. 기존 결과 디렉터리는 덮어쓰지 않습니다. 이 단계는 8.1 실행 없이도 C의 v2 condition과 저장된 S/L scores로 실행할 수 있습니다.

In [10]:
from research.experiments.fiqa_split_stability import (
    DEFAULT_PARTITION_SEEDS, run_fiqa_split_stability, write_fiqa_split_stability,
)
RUN_SPLIT_STABILITY = True
WRITE_SPLIT_STABILITY = True
SPLIT_STABILITY_SEEDS = DEFAULT_PARTITION_SEEDS
SPLIT_STABILITY_RESAMPLES = 2000
SPLIT_STABILITY_BOOTSTRAP_SEED = 8972
split_stability = None
if WRITE_SPLIT_STABILITY and not RUN_SPLIT_STABILITY:
    raise ValueError('WRITE requires RUN_SPLIT_STABILITY=True')
if RUN_SPLIT_STABILITY:
    if condition_tables is None:
        raise RuntimeError('먼저 C에서 v2 condition artifact를 준비하세요.')
    stability_fiqa = {
        variant: load_fiqa_score_artifact(
            RESULT_ROOT / 'fiqa_scores' / 'survface' / CRFIQA_VARIANTS[variant].model_uid
        ) for variant in ('S', 'L')
    }
    split_stability = run_fiqa_split_stability(
        condition_tables, stability_fiqa['S'], stability_fiqa['L'],
        partition_seeds=SPLIT_STABILITY_SEEDS, target_fpirs=TARGET_FPIRS,
        safety_fraction=SAFETY_FRACTION, bin_count=QUALITY_BIN_COUNT,
        shrinkage_strength=SHRINKAGE_STRENGTH,
        minimum_group_non_mated=MINIMUM_GROUP_NON_MATED,
        resamples=SPLIT_STABILITY_RESAMPLES, bootstrap_seed=SPLIT_STABILITY_BOOTSTRAP_SEED,
        progress=lambda status: print(status, flush=True),
    )
    if WRITE_SPLIT_STABILITY:
        split_stability_path = write_fiqa_split_stability(
            RESULT_ROOT / 'fiqa_split_stability' / str(source_manifest['run_id']),
            split_stability,
        )
        display(Markdown(f'새 분할 안정성 결과: `{split_stability_path}`'))
    display(split_stability['stability_summary'][[
        'target_fpir', 'method', 'split_count', 'target_met_split_count',
        'fpir_min', 'fpir_median', 'fpir_max', 'tpir_median', 'observed_pattern',
    ]])
else:
    display(Markdown('분할 안정성 대기: RUN/WRITE를 명시적으로 활성화하세요.'))

{'completed': 1, 'total': 20, 'partition_seed': 0}
{'completed': 2, 'total': 20, 'partition_seed': 1}
{'completed': 3, 'total': 20, 'partition_seed': 2}
{'completed': 4, 'total': 20, 'partition_seed': 3}
{'completed': 5, 'total': 20, 'partition_seed': 4}
{'completed': 6, 'total': 20, 'partition_seed': 5}
{'completed': 7, 'total': 20, 'partition_seed': 6}
{'completed': 8, 'total': 20, 'partition_seed': 7}
{'completed': 9, 'total': 20, 'partition_seed': 8}
{'completed': 10, 'total': 20, 'partition_seed': 9}
{'completed': 11, 'total': 20, 'partition_seed': 10}
{'completed': 12, 'total': 20, 'partition_seed': 11}
{'completed': 13, 'total': 20, 'partition_seed': 12}
{'completed': 14, 'total': 20, 'partition_seed': 13}
{'completed': 15, 'total': 20, 'partition_seed': 14}
{'completed': 16, 'total': 20, 'partition_seed': 15}
{'completed': 17, 'total': 20, 'partition_seed': 16}
{'completed': 18, 'total': 20, 'partition_seed': 17}
{'completed': 19, 'total': 20, 'partition_seed': 18}
{'completed'

새 분할 안정성 결과: `C:\ronbun\results\calibration\fiqa_split_stability\20260830-R001-ec6e5d4a\fiqa-split-fad8e7226da6f1feaffd17fe`

,target_fpir,method,split_count,target_met_split_count,fpir_min,fpir_median,fpir_max,tpir_median,observed_pattern
0,0.01,fiqa_l,20,20,0.007122,0.009254,0.009660,0.001854,all_observed_splits_meet
1,0.05,fiqa_l,20,20,0.037935,0.045093,0.045977,0.006670,all_observed_splits_meet
2,0.10,fiqa_l,20,20,0.077077,0.088643,0.091337,0.012752,all_observed_splits_meet
3,0.20,fiqa_l,20,20,0.162581,0.180678,0.184169,0.024775,all_observed_splits_meet
4,0.30,fiqa_l,20,20,0.249803,0.275900,0.279104,0.038330,all_observed_splits_meet
5,0.01,fiqa_s,20,20,0.007639,0.009377,0.009726,0.001978,all_observed_splits_meet
6,0.05,fiqa_s,20,20,0.039750,0.046248,0.047619,0.006719,all_observed_splits_meet
7,0.10,fiqa_s,20,20,0.080970,0.090598,0.092766,0.012115,all_observed_splits_meet
8,0.20,fiqa_s,20,20,0.163731,0.181060,0.184670,0.023178,all_observed_splits_meet
9,0.30,fiqa_s,20,20,0.248907,0.272791,0.276582,0.035144,all_observed_splits_meet


## 8.3 완료 결과의 통합 보고 — 재실험 없음

공통 보고 노트북과 같은 검증·표 생성 모듈을 사용합니다. 8.1/8.2의 RUN이 False여도 명시적으로 고정한 완료 artifact를 읽습니다. source run/model/condition이 일치해야 하며 오래된 지표, 누락·변조 결과는 중단합니다. 기본 읽기 True·저장 False입니다. 실제 FPIR 실패, 고정 threshold CI, seed 변동을 구분해 읽으세요.

In [11]:
from research.experiments.calibration_evidence_report import (
    pinned_report_sources, load_calibration_evidence_report,
    render_calibration_evidence_markdown, write_calibration_evidence_report,
)
LOAD_INTEGRATED_EVIDENCE_REPORT = False
WRITE_INTEGRATED_EVIDENCE_REPORT = False
INTEGRATED_EVIDENCE_SOURCES = pinned_report_sources(PROJECT_ROOT)
integrated_evidence_report = None
if WRITE_INTEGRATED_EVIDENCE_REPORT and not LOAD_INTEGRATED_EVIDENCE_REPORT:
    raise ValueError('WRITE requires LOAD_INTEGRATED_EVIDENCE_REPORT=True')
if LOAD_INTEGRATED_EVIDENCE_REPORT:
    if condition_tables is None:
        raise RuntimeError('먼저 C의 v2 condition을 준비하세요.')
    integrated_evidence_report = load_calibration_evidence_report(
        **INTEGRATED_EVIDENCE_SOURCES, expected_run_id=str(source_manifest['run_id']),
        expected_model_uid=condition_tables.manifest['model_uid'], reference_seed=8972,
    )
    if integrated_evidence_report['manifest']['condition_uid'] != condition_tables.condition_uid:
        raise ValueError('현재 notebook condition과 보고 condition이 다릅니다.')
    display(Markdown(render_calibration_evidence_markdown(integrated_evidence_report)))
    if WRITE_INTEGRATED_EVIDENCE_REPORT:
        print(write_calibration_evidence_report(
            PROJECT_ROOT / 'results/paper/common/calibration_evidence', integrated_evidence_report))

## 9. Saliency 1차 목적 — 기존 압축/retrieval 진단 유지

이 셀은 새 threshold를 학습하지 않습니다. 기존 ArcFace/SurvFace 결과에서 PQ m128 ADC 조건의 공간적 saliency feature와 embedding distortion, score/rank 변화, crossing 간 연관 결과를 읽습니다. Spearman rho는 보조 진단이며 임의 threshold 가중식으로 바꾸지 않습니다.

In [12]:
primary_saliency_views = {}
if RUN_PRIMARY_SALIENCY_SUMMARY:
    primary_saliency = load_saliency_primary_diagnostics(SOURCE_RUN_DIR)
    for name, frame in primary_saliency.items():
        mask = pd.Series(True, index=frame.index)
        if 'compression_profile' in frame:
            mask &= frame['compression_profile'].astype(str).eq(COMPRESSION_PROFILE)
        if 'search_mode' in frame:
            mask &= frame['search_mode'].astype(str).eq(SEARCH_MODE)
        selected = frame.loc[mask].copy()
        preferred = [
            'analysis_scope', 'analysis_tier', 'compression_profile', 'search_mode',
            'target_fpir', 'threshold_policy', 'is_mated', 'saliency_feature',
            'instability_predictor', 'sensitivity_metric', 'event_metric',
            'sample_count', 'paired_query_count', 'identity_count', 'event_count',
            'event_rate', 'spearman_rho', 'bootstrap_ci_low', 'bootstrap_ci_high',
            'frozen_event_count', 'frozen_event_rate', 'recalibrated_event_count',
            'recalibrated_event_rate', 'recalibrated_minus_frozen_rate',
            'resolved_event_count', 'introduced_event_count',
            'frozen_spearman_rho', 'recalibrated_spearman_rho',
            'recalibrated_minus_frozen_rho', 'paired_bootstrap_ci_low',
            'paired_bootstrap_ci_high', 'event_support_eligible',
            'association_status',
        ]
        columns = [column for column in preferred if column in selected]
        primary_saliency_views[name] = selected[columns].reset_index(drop=True)
        display(Markdown(f'**{name}** — {len(selected):,} rows'))
        if (
            selected.empty
            and SEARCH_MODE == 'pq_adc_exhaustive'
            and name in {'threshold_policy', 'threshold_policy_rho'}
        ):
            display(Markdown('PQ ADC score-space에는 frozen-origin 대 recalibrated threshold policy 비교가 적용되지 않습니다(not applicable).'))
            continue
        preview = primary_saliency_views[name]
        if 'saliency_feature' in preview and not preview.empty:
            preview = (
                preview.sort_values('saliency_feature')
                .groupby('saliency_feature', sort=True, group_keys=False)
                .head(1)
                .reset_index(drop=True)
            )
        display(preview.head(30))
else:
    display(Markdown('`RUN_PRIMARY_SALIENCY_SUMMARY=False`: 기존 saliency 진단 로드를 건너뜁니다.'))

`RUN_PRIMARY_SALIENCY_SUMMARY=False`: 기존 saliency 진단 로드를 건너뜁니다.

## 10. Saliency 2차 목적 — FIQA 이후 추가정보 검증 readiness gate

사전 지정 feature는 `outside_face_attention`, `saliency_entropy` 두 개뿐입니다. `Random`은 threshold feature가 아니라 faithfulness negative control로만 사용합니다. 강한 reliability gate는 identity-cluster bootstrap에서 `High−Low`와 `High−Random`의 CI 하한이 모두 0보다 클 때만 통과합니다. 하나라도 실패하면 gated High/Low contrast는 0입니다. 이 gate와 별개로 calibration/test 모두에서 동일 target의 saliency coverage가 95% 이상이어야 다음 노트북(`01_saliency_incremental_threshold_calibration.ipynb`, 향후 작성 대상)으로 진행할 수 있습니다. 현재 artifact는 test-only이므로 실제 FIQA+Saliency calibration은 차단됩니다.

In [13]:
saliency_readiness = None
faithfulness_reliability = None
faithfulness_summary = pd.DataFrame()
saliency_correction_enabled = False
saliency_reliability_weight = 0.0
SALIENCY_FEATURE_PATH = (
    SOURCE_RUN_DIR / 'artifacts' / 'step2_workflow'
    / 'saliency_population' / 'saliency_features.csv'
)
if RUN_SALIENCY_READINESS_CHECK and condition_tables is not None:
    faithfulness_maximum_samples = resolve_common_faithfulness_maximum_samples(
        PROJECT_ROOT,
        datasets=('survface',),
        run_ids={'survface': str(source_manifest['run_id'])},
    )
    faithfulness_artifacts = load_selected_faithfulness_artifacts(
        PROJECT_ROOT,
        datasets=('survface',),
        model_uids={'survface': source_model_uid},
        run_ids={'survface': str(source_manifest['run_id'])},
        maximum_samples=faithfulness_maximum_samples,
    )
    faithfulness_summary = faithfulness_artifacts.summary
    faithfulness_reliability = assess_saliency_faithfulness_reliability(
        faithfulness_summary, group='all'
    )
    display(
        faithfulness_summary.loc[
            faithfulness_summary['group'].astype(str).eq('all'),
            ['metric', 'sample_count', 'identity_count', 'mean', 'mean_ci_lower', 'mean_ci_upper'],
        ].reset_index(drop=True)
    )
    display(pd.DataFrame([faithfulness_reliability.as_dict()]))
    saliency_features = pd.read_csv(
        SALIENCY_FEATURE_PATH,
        usecols=[
            'sample_id', 'saliency_target_name', 'heatmap_available',
            'gradcam_valid_heatmap', 'outside_face_attention', 'saliency_entropy',
        ],
        low_memory=False,
    )
    saliency_readiness = assess_saliency_incremental_readiness(
        condition_tables.calibration,
        condition_tables.test,
        saliency_features,
        requested_features=('outside_face_attention', 'saliency_entropy'),
        minimum_coverage=0.95,
    )
    display(pd.DataFrame([saliency_readiness.as_dict()]))
    saliency_correction_enabled = bool(
        saliency_readiness.secondary_calibration_supported
        and faithfulness_reliability.strong_faithfulness_pass
    )
    saliency_reliability_weight = (
        faithfulness_reliability.gated_high_low_contrast
        if saliency_correction_enabled
        else 0.0
    )
    if not faithfulness_reliability.strong_faithfulness_pass:
        display(Markdown('**차단됨:** High가 Low와 Random을 모두 이기지 못했습니다. Random은 negative control로만 보고하며 saliency correction은 0입니다.'))
    if not saliency_readiness.secondary_calibration_supported:
        display(Markdown('**차단됨:** calibration saliency 없이 FIQA+Saliency threshold를 fit하면 test leakage가 되므로 saliency correction은 0입니다.'))
elif RUN_SALIENCY_READINESS_CHECK:
    display(Markdown('**대기/차단:** v2 condition score가 없습니다. C 단계 준비 전까지 FIQA+Saliency는 비활성 상태입니다.'))
else:
    display(Markdown('`RUN_SALIENCY_READINESS_CHECK=False`: faithfulness/Random gate와 대용량 saliency feature 파일을 읽지 않습니다. 2차 calibration은 차단 상태입니다.'))

`RUN_SALIENCY_READINESS_CHECK=False`: faithfulness/Random gate와 대용량 saliency feature 파일을 읽지 않습니다. 2차 calibration은 차단 상태입니다.

In [14]:
# 11. 현재 상태 요약 — 계산 완료와 미실행을 명확히 구분
status_rows = [
    {'stage': 'checkpoint_preflight', 'status': 'validated', 'artifact': str(FIQA_CHECKPOINTS[FIQA_VARIANT])},
    {'stage': 'fiqa_scores', 'status': 'available' if fiqa_artifact is not None else 'not_run', 'action': fiqa_stage_action, 'artifact': str(FIQA_OUTPUT_DIR)},
    {'stage': 'calibration_score_replay', 'status': 'available' if condition_tables is not None else 'not_run', 'action': condition_stage_action, 'artifact': str(CONDITION_OUTPUT_DIR)},
    {'stage': 'global_vs_fiqa', 'status': 'available' if comparison is not None else 'not_run', 'action': calibration_stage_action, 'artifact': str(CALIBRATION_OUTPUT_DIR)},
    {'stage': 'saliency_primary_analysis', 'status': 'available' if primary_saliency_views else 'not_loaded', 'artifact': str(SOURCE_RUN_DIR / 'artifacts' / 'step2_workflow')},
    {
        'stage': 'saliency_faithfulness_random_control',
        'status': faithfulness_reliability.status if faithfulness_reliability is not None else 'not_checked',
        'action': f'gated_contrast={saliency_reliability_weight:.12g}',
        'artifact': str(faithfulness_artifacts.roots['survface']) if faithfulness_reliability is not None else 'not_loaded',
    },
    {
        'stage': 'fiqa_plus_saliency',
        'status': 'ready' if saliency_correction_enabled else 'blocked',
        'action': 'enabled' if saliency_correction_enabled else 'correction_zero',
        'artifact': 'not_created',
    },
]
status_table = pd.DataFrame(status_rows)
display(status_table)

,stage,status,artifact,action
0,checkpoint_preflight,validated,C:\ronbun\models\fiqa\CR-FIQA(L).pth,NaN
1,fiqa_scores,available,C:\ronbun\results\calibration\fiqa_scores\surv...,loaded_verified
2,calibration_score_replay,available,C:\ronbun\results\calibration\condition_scores...,computed_written
3,global_vs_fiqa,available,C:\ronbun\results\calibration\global_vs_fiqa\2...,computed_written
4,saliency_primary_analysis,not_loaded,C:\ronbun\runs\survface_20260830\20260830-R001...,NaN
5,saliency_faithfulness_random_control,not_checked,not_loaded,gated_contrast=0
6,fiqa_plus_saliency,blocked,not_created,correction_zero


## 12. 해석 규칙

- 결론은 `realized_fpir`, Wilson 95% CI, `tpir_at_rank_k`(현재 Rank-20), `target_met_on_test`, paired bootstrap 차이를 함께 봅니다.
- TPIR20은 정답 identity의 점수 자체가 threshold 이상이고 정답 rank가 20 이내인 비율입니다. Top-1 acceptance로 대체하지 않습니다.
- v1 결과는 보존하되 자동 재사용하지 않습니다. C의 RUN/WRITE를 켜면 기존 calibration 점수와 검증된 test ledger를 재사용해 별도 v2 경로를 생성합니다. 이어 D의 RUN/WRITE를 켜 비교 결과를 재생성합니다.
- S/L 비교에는 같은 PARTITION_SEED와 보정 설정을 사용합니다. 현재 paired CI는 query 단위·고정 threshold의 탐색적 CI이며 identity 상관, calibration 재적합 변동과 다중 비교를 보정하지 않습니다.
- `global_safe`와 `shrunk_safe`는 held-out 경험적 보수화이며 formal FPIR guarantee가 아닙니다.
- FIQA가 아주 조금 좋아졌다는 이유만으로 채택하지 않습니다. 사전 지정한 여러 target FPIR에서 방향이 재현되고, paired CI와 TPIR 손실까지 검토해야 합니다.
- FIQA가 Global보다 낫지 않아도 saliency의 1차 연구 질문은 독립적으로 유지됩니다.
- Random은 calibration/threshold feature가 아니라 필수 faithfulness negative control입니다. `High−Low`와 `High−Random` paired CI 하한이 모두 양수일 때만 strong faithfulness를 통과합니다.
- strong faithfulness 또는 calibration/test saliency coverage 중 하나라도 실패하면 `saliency_correction_enabled=False`, `saliency_reliability_weight=0`으로 유지합니다.
- FIQA+Saliency는 calibration saliency를 별도로 생성한 뒤에만 시험하며, test 기반 feature 선택이나 threshold 재조정은 금지합니다.
- 이 노트북에서 생성한 compact artifact만 이후 `00_cross_dataset_results.ipynb`의 입력 후보가 됩니다. 공통 보고 노트북 연결은 결과가 실제로 생성·검증된 뒤 별도 변경으로 수행합니다.